# Install kaggle dataset

In [3]:
pip install kagglehub


   ---------------------------------------- 0.0/68.3 kB ? eta -:--:--
   ------------------------ --------------- 41.0/68.3 kB 991.0 kB/s eta 0:00:01
   ---------------------------------------- 68.3/68.3 kB 935.8 kB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [53]:
import kagglehub

path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")

print("Path to dataset files:", path)

import pandas as pd
true_dataset_path = path + '\True.csv'
fake_dataset_path = path + "\Fake.csv"

df_true = pd.read_csv(true_dataset_path)
df_fake = pd.read_csv(fake_dataset_path)

Path to dataset files: C:\Users\maria\.cache\kagglehub\datasets\clmentbisaillon\fake-and-real-news-dataset\versions\1


# Combine datasets

In [54]:
# Add label columns
df_true['label'] = 1  
df_fake['label'] = 0  

# Combine the datasets
df_true_fake = pd.concat([df_true, df_fake], ignore_index=True)

# Shuffle
df_true_fake = df_true_fake.sample(frac=1, random_state=42).reset_index(drop=True)


# Installations

In [21]:
!pip install transformers datasets accelerate -q
!pip install huggingface_hub


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
!pip install bertopic -q


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
import pandas as pd
import numpy as np
import re

# Help functions

In [40]:
def clean_date(d):
  d = str(d).strip()
  if d.startswith('http') or d == '' or len(d) > 20 or d == 'nan':
    return np.nan
  try:
      return pd.to_datetime(d).strftime('%Y-%m-%d')
  except:
      return np.nan

In [41]:
def clean_text(text):
  text = str(text)
  text = re.sub(r'^[A-Z][a-z]+ [A-Z][a-z]+ says\s*', '', text)
  text = re.sub(r'Featured image.*$', '', text, flags=re.I)
  text = re.sub(r'Image source.*$', '', text, flags=re.I)
  text = re.sub(r'\s+', ' ', text).strip()
  return text

# Data preprocesing

In [55]:
print(len(df_true_fake))
df_true_fake = df_true_fake.drop_duplicates(subset=['title', 'text']).reset_index(drop=True)
print(len(df_true_fake))

44898
39105


In [56]:
df_true_fake['date'] = df_true_fake['date'].apply(clean_date)
df_true_fake = df_true_fake.dropna(subset=['date']).reset_index(drop=True)

df_true_fake['text_clean'] = df_true_fake['text'].apply(clean_text)
df_true_fake['title_clean'] = df_true_fake['title'].apply(clean_text)

df_true_fake['content'] = df_true_fake['title_clean'] + " " + df_true_fake['text_clean']

In [32]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np


In [57]:
# Run BERTopic separately first
print("\nRunning BERTopic...")
topic_model = BERTopic(language="english", verbose=True)
texts = df_true_fake[text_column].fillna('').astype(str).tolist()
topics, _ = topic_model.fit_transform(texts)

# Get topic probabilities for each document
topic_info = topic_model.get_document_info(texts)
topic_probabilities = topic_model.approximate_distribution(texts)



2025-12-20 20:57:24,180 - BERTopic - Embedding - Transforming documents to embeddings.



Running BERTopic...


Batches: 100%|██████████| 1222/1222 [40:45<00:00,  2.00s/it]
2025-12-20 21:38:18,276 - BERTopic - Embedding - Completed ✓
2025-12-20 21:38:18,279 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-12-20 21:38:43,579 - BERTopic - Dimensionality - Completed ✓
2025-12-20 21:38:43,588 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-12-20 21:38:56,983 - BERTopic - Cluster - Completed ✓
2025-12-20 21:38:57,121 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-12-20 21:39:23,478 - BERTopic - Representation - Completed ✓
100%|██████████| 40/40 [19:23<00:00, 29.10s/it]


In [62]:
model_dir = "bert_fake_true"
topic_model.save(model_dir, save_embedding_model=True, save_ctfidf=True,
                      serialization="pytorch")

from huggingface_hub import upload_folder

if hasattr(topic_model.ctfidf_model, '_idf_diag'):
    import scipy.sparse as sp
    idf_diag = topic_model.ctfidf_model._idf_diag

    if sp.issparse(idf_diag):
        idf_values = idf_diag.diagonal()
    else:
        idf_values = idf_diag

    topic_model.ctfidf_model._idf_diag = idf_values

upload_folder(
    folder_path=model_dir,
    repo_id="rasm01/bert_fake_true",
    commit_message="Full BERTopic model upload with all files",
)

print("Загрузка завершена! Теперь BERTopic.load сработает.")

ctfidf.bin:   0%|          | 0.00/22.9M [00:00<?, ?B/s]

ctfidf.bin:   0%|          | 16.4k/22.9M [00:00<19:37, 19.5kB/s]
ctfidf.bin:   2%|▏         | 475k/22.9M [00:01<00:32, 695kB/s]  
ctfidf.bin:   3%|▎         | 721k/22.9M [00:01<00:22, 1.01MB/s]
topic_embeddings.bin: 100%|██████████| 880k/880k [00:01<00:00, 473kB/s] 
ctfidf.bin: 100%|██████████| 22.9M/22.9M [00:07<00:00, 2.98MB/s]
Upload 2 LFS files: 100%|██████████| 2/2 [00:08<00:00,  4.26s/it]


Загрузка завершена! Теперь BERTopic.load сработает.


### Create topic mapping

In [64]:
pip install openai

  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached jiter-0.12.0-cp311-cp311-win_amd64.whl.metadata (5.3 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.1/1.1 MB 1.3 MB/s eta 0:00:01
   ---------- ----------------------------- 0.3/1.1 MB 2.2 MB/s eta 0:00:01
   ---------------- ----------------------- 0.5/1.1 MB 2.6 MB/s eta 0:00:01
   ----------------------------- ---------- 0.8/1.1 MB 3.6 MB/s eta 0:00:01
   ---------------------------------------  1.1/1.1 MB 4.0 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 3.6 MB/s eta 0:00:00
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached jiter-0.12.0-cp311-cp311-win_amd64.whl (204 kB)
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
Note: you may need to restart the kernel to


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import openai
import json

# === SET YOUR API KEY HERE ===
OPENAI_API_KEY = "_"
# =============================

def create_simple_topic_mapping(topic_model, filename="topic_mapping.json"):
    """
    Create simple topic mapping using OpenAI API.
    
    Args:
        topic_model: Your trained BERTopic model
        filename: Where to save the JSON mapping
    """
    
    # Initialize OpenAI
    client = openai.OpenAI(api_key=OPENAI_API_KEY)
    
    # Get all topic IDs (skip -1 which is outliers)
    topic_ids = sorted(set(topic_model.topics_))
    topic_ids = [t for t in topic_ids if t != -1]
    
    print(f"Found {len(topic_ids)} topics")
    
    # Create mapping dictionary
    mapping = {}
    
    for topic_id in topic_ids:
        print(f"Naming topic {topic_id}...")
        
        # Get top keywords for this topic
        try:
            topic_words = topic_model.get_topic(topic_id)
            if topic_words:
                keywords = [word for word, _ in topic_words[:10]]
                keyword_text = ", ".join(keywords)
            else:
                keyword_text = "no keywords found"
        except:
            keyword_text = "error getting keywords"
        
        # Ask OpenAI for a simple name
        try:
            response = client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[
                    {"role": "system", "content": "You create short, descriptive topic names (2-3 words)."},
                    {"role": "user", "content": f"Create a short topic name for texts about: {keyword_text}. Respond with just the name."}
                ],
                temperature=0.3,
                max_tokens=15
            )
            
            topic_name = response.choices[0].message.content.strip()
            topic_name = topic_name.strip('"\'')
            print(f"  -> {topic_name}")
            
        except Exception as e:
            print(f"  -> Error: {e}. Using fallback name.")
            topic_name = f"topic_{topic_id}"
        
        mapping[topic_id] = topic_name
        # Small delay to avoid rate limits
        import time
        time.sleep(0.5)
    
    # Add outliers topic
    mapping[-1] = "outliers"
    
    # Save to JSON file
    with open(filename, 'w') as f:
        json.dump(mapping, f, indent=2)
    
    print(f"\n✅ Saved mapping to {filename}")
    return mapping



In [66]:

mapping = create_simple_topic_mapping(topic_model)

Found 571 topics
Naming topic 0...
  -> Healthcare Reform
Naming topic 1...
  -> Russian Cyber Election Interference
Naming topic 2...
  -> Islamic Extremism
Naming topic 3...
  -> Corporate Tax Reform
Naming topic 4...
  -> Bite-sized Wisdom
Naming topic 5...
  -> FBI Email Investigation
Naming topic 6...
  -> Myanmar Rohingya Crisis
Naming topic 7...
  -> Iran Nuclear Deal
Naming topic 8...
  -> Police shootings in Ferguson
Naming topic 9...
  -> Catalan Independence
Naming topic 10...
  -> Gun Control Legislation
Naming topic 11...
  -> Abortion and Parenthood
Naming topic 12...
  -> Brexit Negotiations
Naming topic 13...
  -> Protest Rally Dynamics
Naming topic 14...
  -> Sweden-Germany Relations
Naming topic 15...
  -> Legal Travel Restrictions
Naming topic 16...
  -> Xi Jinping's China
Naming topic 17...
  -> Comey Investigation
Naming topic 18...
  -> Supreme Court Nominee Vacancy
Naming topic 19...
  -> Kurdish Politics
Naming topic 20...
  -> Venezuelan Politics
Naming topic 2

# TF-IDF + Logistic Regression

In [46]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import joblib

In [59]:
# Use 'text' or 'content' column (adjust based on your dataset)
text_column = 'text' if 'text' in df_true_fake.columns else 'content'
print(f"Using '{text_column}' column for text")

# PREPARE TRAINING DATA
X = df_true_fake[text_column].fillna('')
y = df_true_fake['label']

print(f"\nDataset stats:")
print(f"Total samples: {len(df_true_fake)}")
print(f"True news: {(y == 1).sum()} ({(y == 1).sum()/len(y)*100:.1f}%)")
print(f"Fake news: {(y == 0).sum()} ({(y == 0).sum()/len(y)*100:.1f}%)")



Using 'text' column for text

Dataset stats:
Total samples: 39099
True news: 21197 (54.2%)
Fake news: 17902 (45.8%)


In [68]:
# 3. TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")

# 4. TRAIN MODEL
print("\nTraining model...")
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
classifier = LogisticRegression(max_iter=1000, random_state=42)

# Transform text to features
X_train_vec = vectorizer.fit_transform(X_train)

# Train classifier
classifier.fit(X_train_vec, y_train)

# 5. EVALUATE
from sklearn.metrics import accuracy_score

X_test_vec = vectorizer.transform(X_test)
y_pred = classifier.predict(X_test_vec)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nModel Accuracy: {accuracy:.2%}")


Training set: 31279 samples
Test set: 7820 samples

Training model...

Model Accuracy: 98.43%


### Save log_reg an vectorizer to huggin face repo

In [63]:
model_dir = "log_reg_fake_true"
print(f"\nSaving model to '{model_dir}'...")

import pickle
import os

# Create directory
os.makedirs(model_dir, exist_ok=True)

# Save vectorizer
with open(os.path.join(model_dir, "vectorizer.pkl"), "wb") as f:
    pickle.dump(vectorizer, f)

# Save classifier
with open(os.path.join(model_dir, "classifier.pkl"), "wb") as f:
    pickle.dump(classifier, f)

print(f"✅ Model saved to '{model_dir}' folder")

# 3. Upload to Hugging Face
from huggingface_hub import upload_folder

print(f"\nUploading to Hugging Face...")
upload_folder(
    folder_path=model_dir,
    repo_id="rasm01/log_reg_fake_true",  # Your repo name
    commit_message=f"Logistic Regression model for fake news detection (Accuracy: {accuracy:.2%})",
)

print("✅ Model uploaded to Hugging Face!")
print(f"📦 Available at: https://huggingface.co/rasm01/log_reg_fake_true")


Saving model to 'log_reg_fake_true'...
✅ Model saved to 'log_reg_fake_true' folder

Uploading to Hugging Face...


classifier.pkl:   0%|          | 0.00/40.7k [00:00<?, ?B/s]
classifier.pkl:  40%|████      | 16.4k/40.7k [00:00<00:01, 24.0kB/s]

classifier.pkl: 100%|██████████| 40.7k/40.7k [00:01<00:00, 37.5kB/s]
vectorizer.pkl: 100%|██████████| 184k/184k [00:01<00:00, 154kB/s]
Upload 2 LFS files: 100%|██████████| 2/2 [00:02<00:00,  1.00s/it]


✅ Model uploaded to Hugging Face!
📦 Available at: https://huggingface.co/rasm01/log_reg_fake_true
